vamos a ver si podemos usar el rag de azure desde este jupyter notebook que nunca he usado

In [ ]:
pip install openai requests python-dotenv azure-search-documents azure-core

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
chat_gpt_api_key = os.getenv("CHAT_GPT_4_API_KEY")
chat_gpt_api_model = os.getenv("CHAT_GPT_4_MODEL")
chat_gpt_api_endpoint = os.getenv("CHAT_GPT_4_ENDPOINT")
chat_gpt_api_temperature = float(os.getenv("CHAT_GPT_4_TEMPERATURE", 0.7))
chat_gpt_api_system_prompt = os.getenv("CHAT_GPT_4_SYSTEM_PROMPT", "You are a helpful assistant.")
chat_gpt_api_version = os.getenv("CHAT_GPT_4_API_VERSION")
index_name = os.getenv("INDEX_NAME")
index_endpoint = os.getenv("INDEX_ENDPOINT")
index_key= os.getenv("INDEX_KEY")
print(f'cargadas las variables de entorno de chat gpt 4')

In [ ]:
embedding_api_key = os.getenv("EMBEDDING_API_KEY")
embedding_api_endpoint = os.getenv("EMBEDDING_API_ENDPOINT")
embedding_api_model = os.getenv("EMBEDDING_API_MODEL")

print(f'cargadas las variables de entorno de embedding')

ahora con todo cargado tengo que hacer primero una al vector para tener el valor vectorizado  de la consulta

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=chat_gpt_api_key,
    api_version="2023-05-15",
    azure_endpoint=chat_gpt_api_endpoint
)



ahora tengo que crear  un nuevo embedding desde el cliente de azure que ha creado antes

In [ ]:
def generate_embeddings(client, text):
    response = client.embeddings.create(
        input=text,
        model = "text-embedding-ada-002"
    )
    embeddings=response.model_dump()
    print(f'Embeddings generados {embeddings}', embeddings)
    return embeddings['data'][0]['embedding']

ahora llamo al api para traer el asunto  de la vectorizacion

In [ ]:
user_query = input()
vectorised_user_query = generate_embeddings(azure_openai_client, user_query)
print(vectorised_user_query)
context=[]

#vale ya tengo el vectorizado del resultado de la consulta ahora tengo que hacer la consulta al indice y despues al llm

In [ ]:
from azure.search.documents.models import VectorizedQuery
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

credential = AzureKeyCredential(index_key)

search_client = SearchClient(endpoint=index_endpoint, index_name=index_name, credential=credential)

vector_query = VectorizedQuery(
    vector=vectorised_user_query,
    k_nearest_neighbors=50,
    fields="text_vector"
)

try:
    # 3. Realizar la Búsqueda Híbrida
    response = search_client.search(
        search_text=user_query,  # <--- PARTE TRADICIONAL (Keyword)
        vector_queries=[vector_query],
        select=["*"],
        top=5
    )
    for doc in response:
        context.append(dict(
        {
            "chunk": doc['chunk'],
            "score": doc['@search.score']

        }
        ))
except Exception as e:
    print('request exception', e)


In [ ]:
system_prompt = f""""Eres un dungeon master que lleva toda la vida jugando a advanced dungeons and dragons e intentas que parezca misterioso el asunto"""

user_prompt = f""" the user query is: {user_query}
the context is : {context}"""

print(user_prompt)

chat_completions_response = azure_openai_client.chat.completions.create(
    model = chat_gpt_api_model,
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.7
)
print("***********************")
print(chat_completions_response.choices[0].message.content)